In [ ]:
!pip install -q -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

In [ ]:
import os
from google.colab import userdata, files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [ ]:
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("Gemini_API_Key_2")
if not api_key:
    try:
        api_key = userdata.get("Gemini_API_Key")
    except Exception:
        pass
    os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

In [ ]:
print(" Please upload one or more PDF files:")
uploaded = files.upload()

pdf_texts = []
for filename in uploaded.keys():
    if filename.endswith(".pdf"):
        reader = PdfReader(filename)
        text = ""
        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n" + page_text
        pdf_texts.append(text)
        print(f" Loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
    raise ValueError("No valid PDF files uploaded. Please re-run and upload a .pdf file.")

full_pdf_content = "\n\n".join(pdf_texts)

 Please upload one or more PDF files:


In [ ]:
print(" Loading embedding model and building vector index...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

# Reset collection for clean execution
try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

collection = chroma_client.create_collection(name="pdf_rag_collection")

# Embed chunks in batches
chunk_embeddings = embedder.encode(chunks).tolist()
chunk_ids = [f"doc_chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunk_ids
)
print(" PDF Vector Indexing Complete!\n")

In [ ]:
def retrieve_pdf_context(query: str, top_k: int = 3) -> list[str]:
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0]

def ask_pdf(query: str):
    context_passages = retrieve_pdf_context(query, top_k=3)
    context_str = "\n".join(f"- {p}" for p in context_passages)

    prompt = f"""You are an intelligent document analysis assistant. Answer the question using ONLY the provided PDF context below.
If the information is not contained within the provided context, state clearly: "I cannot find the answer in the provided PDF."

PDF Context:
{context_str}

Question: {query}
Answer:"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, context_passages

In [ ]:
print("=" * 60)
print(" PDF CHATBOT READY! Type your question below (or type 'exit' to quit).")
print("=" * 60)

while True:
    user_query = input("\nAsk a question about your PDF: ")
    if user_query.lower() in ["exit", "quit", "q"]:
        print(" Exiting PDF Chatbot. Goodbye!")
        break
    if not user_query.strip():
        continue

    answer, context = ask_pdf(user_query)

    print("\n--- RETRIEVED PDF SNIPPETS ---")
    for i, snippet in enumerate(context, 1):
        print(f"[{i}] {snippet[:150]}...")

    print("\n--- GEMINI RESPONSE ---")
    print(answer)
    print("-" * 60)

In [ ]:
import os

# 1. Setup Git Repository Structure
os.makedirs("incident_triage_app", exist_ok=True)
os.chdir("incident_triage_app")

!git init
!git config user.name "AI Engineer"
!git config user.email "engineer@example.com"

# 2. Write requirements.txt
with open("requirements.txt", "w") as f:
    f.write("""fastapi
uvicorn
google-genai
pydantic
""")

# 3. Write FastAPI Multi-Agent Application
app_code = '''import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from google import genai
from google.genai import types

app = FastAPI(title="Multi-Agent IT Incident Triager")

# Initialize Gemini Client
api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key) if api_key else None

class IncidentRequest(BaseModel):
    ticket_id: str
    raw_log: str

class IncidentResponse(BaseModel):
    ticket_id: str
    extracted_error: str
    severity: str
    recommended_action: str

# --- AGENT DEFINITIONS ---

def extractor_agent(raw_log: str) -> str:
    """Agent 1: Extracts structured error details from messy logs."""
    prompt = f"Analyze this log and extract only the root error message and affected service in 2 sentences:\\n{raw_log}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text.strip()

def classifier_agent(error_summary: str) -> str:
    """Agent 2: Evaluates risk and assigns Severity Level (LOW, MEDIUM, HIGH, CRITICAL)."""
    prompt = f"Given this error summary, assign a severity level (LOW, MEDIUM, HIGH, CRITICAL) and state why in 1 sentence:\\n{error_summary}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text.strip()

def resolution_agent(error_summary: str, severity: str) -> str:
    """Agent 3: Recommends immediate remediation steps."""
    prompt = f"Provide a concise 3-step action plan to resolve this issue.\\nError: {error_summary}\\nSeverity: {severity}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text.strip()

# --- MULTI-AGENT ORCHESTRATOR ROUTE ---

@app.post("/triage", response_model=IncidentResponse)
async def triage_incident(request: IncidentRequest):
    if not client:
        raise HTTPException(status_code=500, detail="GEMINI_API_KEY not set in environment.")

    # Multi-Agent Workflow Execution
    extracted_error = extractor_agent(request.raw_log)
    severity_assessment = classifier_agent(extracted_error)
    action_plan = resolution_agent(extracted_error, severity_assessment)

    return IncidentResponse(
        ticket_id=request.ticket_id,
        extracted_error=extracted_error,
        severity=severity_assessment,
        recommended_action=action_plan
    )

@app.get("/")
def health_check():
    return {"status": "Multi-Agent System Operational"}
'''

with open("main.py", "w") as f:
    f.write(app_code)

# 4. Write Dockerfile
dockerfile_code = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile_code)

# 5. Commit to Git
!git add .
!git commit -m "Initial commit: Multi-Agent IT Incident Triage app with Docker and FastAPI"

print("\n--- Git Repository State ---")
!git log --oneline

In [ ]:
import os
import uvicorn
import threading
import time
import requests
from google.colab import userdata

# 1. Ensure API Key is loaded in process environment
os.environ["GEMINI_API_KEY"] = userdata.get("Gemini_API_Key")

# 2. Overwrite main.py with updated model 'gemini-3.6-flash'
app_code = '''import os
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from google import genai

app = FastAPI(title="Multi-Agent IT Incident Triager")

def get_client():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise HTTPException(status_code=500, detail="GEMINI_API_KEY not set in environment.")
    return genai.Client(api_key=api_key)

class IncidentRequest(BaseModel):
    ticket_id: str
    raw_log: str

class IncidentResponse(BaseModel):
    ticket_id: str
    extracted_error: str
    severity: str
    recommended_action: str

# --- AGENT DEFINITIONS (Updated Model: gemini-3.6-flash) ---

def extractor_agent(client, raw_log: str) -> str:
    prompt = f"Analyze this log and extract only the root error message and affected service in 2 sentences:\\n{raw_log}"
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text.strip()

def classifier_agent(client, error_summary: str) -> str:
    prompt = f"Given this error summary, assign a severity level (LOW, MEDIUM, HIGH, CRITICAL) and state why in 1 sentence:\\n{error_summary}"
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text.strip()

def resolution_agent(client, error_summary: str, severity: str) -> str:
    prompt = f"Provide a concise 3-step action plan to resolve this issue.\\nError: {error_summary}\\nSeverity: {severity}"
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    return response.text.strip()

# --- MULTI-AGENT ORCHESTRATOR ROUTE ---

@app.post("/triage", response_model=IncidentResponse)
async def triage_incident(request: IncidentRequest):
    client = get_client()

    extracted_error = extractor_agent(client, request.raw_log)
    severity_assessment = classifier_agent(client, extracted_error)
    action_plan = resolution_agent(client, extracted_error, severity_assessment)

    return IncidentResponse(
        ticket_id=request.ticket_id,
        extracted_error=extracted_error,
        severity=severity_assessment,
        recommended_action=action_plan
    )

@app.get("/")
def health_check():
    return {"status": "Multi-Agent System Operational"}
'''

with open("main.py", "w") as f:
    f.write(app_code)

# 3. Import updated FastAPI application
import importlib
import main
importlib.reload(main)

# 4. Start Uvicorn Server in Background Thread
def run_server():
    uvicorn.run(main.app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Allow server time to boot
time.sleep(3)

# 5. Test Endpoint
payload = {
    "ticket_id": "INC-9021",
    "raw_log": "2026-04-12 10:14:02 ERROR [auth-service] Connection Timeout: Failed to connect to Redis cache at 10.0.4.12:6379 after 5000ms. Database pool exhausted."
}

response = requests.post("http://127.0.0.1:8000/triage", json=payload)

print("\n--- Multi-Agent Triage Pipeline Output ---")
if response.status_code == 200:
    print(response.json())
else:
    print(f"Server Error (Status {response.status_code}):")
    print(response.text)